In [263]:
import pandas as pd
import os
import numpy as np
import re

In [264]:
pd.set_option('display.max_columns', None)

# <span style="color:blue;">**Cancer**</span>

In [265]:
study = "Cancer"

## **STEP 0: Data preparation**

### 1. DICOM

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [266]:
# file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/dicom_tag.xlsx"
file_path = "/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data/dicom_tag.xlsx"
dicom = pd.read_excel(file_path)

In [267]:
dicom.rename(columns={'PatientID': 'PATIENT_STUDY_ID', 'AccessionNumber': 'ACCESSION_NUMBER'}, inplace=True)

In [268]:
dicom["PATIENT_STUDY_ID"].unique().size

5655

In [269]:
PIDs = dicom["PATIENT_STUDY_ID"].unique()

In [270]:
dicom.head(5)

,PATIENT_STUDY_ID,PatientBirthDate,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Series,View,Slab,StudyDescription,SeriesDescription,SeriesNumber,Manufacturer,ManufacturerModelName,SliceThickness,Exposure,Rows,Columns,PixelSpacing,FolderPath
0,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,NaN,SECURE,NaN,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,Hologic R2 ImageChecker CAD SC,1,"R2 Technology, Inc.",Cenova,NaN,NaN,1500.0,1250.0,NaN,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,ML,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R ML,71100000,"HOLOGIC, Inc.",Selenia Dimensions,NaN,102.0,3328.0,2560.0,0.038889\0.038889,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,XCCL,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R XCCL,71100000,"HOLOGIC, Inc.",Selenia Dimensions,NaN,94.0,3328.0,2560.0,0.038889\0.038889,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
3,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,LM,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L LM C-View,71300000,"HOLOGIC, Inc.",Selenia Dimensions,NaN,71.0,2457.0,1890.0,0.087290\0.087290,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
4,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,MLO,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L MLO C-View,71300000,"HOLOGIC, Inc.",Selenia Dimensions,NaN,87.0,2457.0,1890.0,0.086609\0.086609,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


#### Create a list of unique study visit to serve as the reference linking the EHR to the images available for specific patient visits (as images are limited to certain visits, not all).

#### See shared parameters in [parameter exlanation][peid] to link data

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 

In [271]:
# Define the key identifier columns and columns to extract
key_columns = ['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'StudyDate', 'Study', 'Side']
unique_study = dicom.drop_duplicates(subset=key_columns, keep='first')

unique_study = unique_study[key_columns]
unique_study = unique_study[unique_study['Side'].notna()]

unique_study.reset_index(drop=True, inplace=True)

In [272]:
unique_study.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side
0,4330018595,54.0,63737104,2019-08-19,DIAG,R
1,4330018595,54.0,63737104,2019-08-19,DIAG,L
2,4330018595,54.0,60103700,2020-06-02,DIAG,R
3,4330018595,54.0,60690108,2020-06-02,SCREEN,L
4,4330029102,44.0,64888584,2019-05-16,SCREEN,L


In [273]:
unique_study["PATIENT_STUDY_ID"].nunique()

5650

#### Filter study with DBT 

In [274]:
dicom_dbt = dicom[dicom["Series"]=="DBT"]

key_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'Study', 'Side']
unique_study_dicom_dbt = dicom_dbt.drop_duplicates(subset=key_columns, keep='first')
unique_study_dicom_dbt.reset_index(drop=True, inplace=True)

In [275]:
unique_study_dicom_dbt.head(5)

,PATIENT_STUDY_ID,PatientBirthDate,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Series,View,Slab,StudyDescription,SeriesDescription,SeriesNumber,Manufacturer,ManufacturerModelName,SliceThickness,Exposure,Rows,Columns,PixelSpacing,FolderPath
0,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,L,DBT,CC,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.088364\0.088364,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,R,DBT,CC,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.088500\0.088500,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330116791,1963-07-01,56.0,61499674,2019-12-17,DIAG,L,DBT,CC,NaN,DIAG DIG MAMMO LEFT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1996.0,0.106407\0.106407,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
3,4330313855,1980-07-01,37.0,77236131,2018-05-30,DIAG,L,DBT,CC,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.088048\0.088048,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
4,4330313855,1980-07-01,37.0,77236131,2018-05-30,DIAG,R,DBT,CC,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,"HOLOGIC, Inc.",Selenia Dimensions,1.0,NaN,2457.0,1890.0,0.087911\0.087911,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


In [276]:
unique_study_dicom_dbt["PATIENT_STUDY_ID"].nunique()

2433

In [277]:
unique_patient_dbt = unique_study[unique_study["PATIENT_STUDY_ID"].isin(unique_study_dicom_dbt["PATIENT_STUDY_ID"].unique())]

In [278]:
unique_patient_dbt # unique patient that has dbt across stidues

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side
8,4330066079,30.0,61259016,2020-01-14,DIAG,L
9,4330066079,30.0,61259016,2020-01-14,DIAG,R
15,4330116791,56.0,62335422,2019-09-24,DIAG,L
16,4330116791,56.0,61499674,2019-12-17,DIAG,L
77,4330313855,37.0,77236131,2018-05-30,DIAG,L
...,...,...,...,...,...,...
31179,4339601084,46.0,77038224,2018-11-07,DIAG,L
31180,4339601084,46.0,77038224,2018-11-07,DIAG,R
31181,4339601084,46.0,65758752,2019-05-16,DIAG,R
31182,4339601084,47.0,60049293,2020-04-09,SCREEN,L


In [279]:
unique_patient_dbt = pd.merge(
    unique_patient_dbt,
    unique_study_dicom_dbt[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'Study', 'Side', 'Series']],
    on = key_columns,
    how='left'
)

In [280]:
unique_patient_dbt

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Series
0,4330066079,30.0,61259016,2020-01-14,DIAG,L,DBT
1,4330066079,30.0,61259016,2020-01-14,DIAG,R,DBT
2,4330116791,56.0,62335422,2019-09-24,DIAG,L,NaN
3,4330116791,56.0,61499674,2019-12-17,DIAG,L,DBT
4,4330313855,37.0,77236131,2018-05-30,DIAG,L,DBT
...,...,...,...,...,...,...,...
17042,4339601084,46.0,77038224,2018-11-07,DIAG,L,DBT
17043,4339601084,46.0,77038224,2018-11-07,DIAG,R,DBT
17044,4339601084,46.0,65758752,2019-05-16,DIAG,R,DBT
17045,4339601084,47.0,60049293,2020-04-09,SCREEN,L,NaN


### 2. Electric Health Record

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [281]:
# file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/parameters of interest.xlsx"
file_path = "/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data/parameters of interest.xlsx"
file_name = pd.ExcelFile(file_path).sheet_names
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes']

---

# **Extract <span style="color:blue;">MO Cancer**</span> 

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

## **STEP 1**. Merge enteredit_findings with pathology, then merge with patient with DBT available
### (OUTPUT) cancer_cohort

enteredit_findings
* Columns: COMPOSITION_NAME, FINDING_LOCATION, FINDING_CATEGORY, FINDING_REC, EXAM_COMPLETED_DATE
* Cancer: 
    * FINDING_CATEGORY = 4, 4A, 4B, 4C, 5

pathology
* Columns: BX_ID, PATHOLOGY_DATE, LESION_CLASS, SIDE
* Cancer: 
    * LESION_CLASS = 'Malignant'

In [282]:
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes']

In [283]:
fn = file_name[0]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
birads = pd.read_excel(file_path)

In [284]:
birads.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count
0,4330018595,63027507,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-07-26,2
1,4330018595,63737104,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-08-19,1
2,4330018595,63737104,Heterogeneously dense (51% - 75%),NaN,3 - Probably benign - short interval follow-up,F-Follow-up at short interval (1-11 months),2019-08-19,1


In [285]:
birads.drop(columns="FINDING_LOCATION", inplace=True)

In [286]:
fn = file_name[1]
# file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
file_path = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "Cleaned", fn + ".xlsx")
pathology = pd.read_excel(file_path)

In [287]:
pathology.head(3)

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
0,4330018595,487690,2020-06-05,Malignant,R
1,4330018595,475789,2020-07-28,Malignant,R
2,4330029102,477464,2021-05-10,Benign,R


In [288]:
biopsy_categories = [
    '4 - Suspicious abnormality, biopsy should be considered',
    '4A - Suspicious abnormality - biopsy should be considered - low suspicion',
    '4B - Suspicious abnormality - biopsy should be considered - intermediate suspicion',
    '4C - Suspicious abnormality - biopsy should be considered - moderate suspicion',
    '5 - Highly suggestive of malignancy, appropriate action should be taken'
]

In [289]:
# ── STEP 1: Merge birads + pathology ────────────────────────────────────────

birads['EXAM_COMPLETED_DATE'] = pd.to_datetime(birads['EXAM_COMPLETED_DATE'])
pathology['PATHOLOGY_DATE']   = pd.to_datetime(pathology['PATHOLOGY_DATE'])
birads['needs_biopsy']        = birads['FINDING_CATEGORY'].isin(biopsy_categories)

# STEP 1.1: Biopsy findings → link to pathology via date window
birads_biopsy    = birads[birads['needs_biopsy']].copy()
biopsy_with_path = pd.merge(birads_biopsy, pathology, on='PATIENT_STUDY_ID', how='left')
biopsy_with_path = biopsy_with_path[
    (biopsy_with_path['PATHOLOGY_DATE'] >= biopsy_with_path['EXAM_COMPLETED_DATE']) &
    (biopsy_with_path['PATHOLOGY_DATE'] <= biopsy_with_path['EXAM_COMPLETED_DATE'] + pd.DateOffset(months=6))
]
biopsy_with_path['days_diff'] = (
    biopsy_with_path['PATHOLOGY_DATE'] - biopsy_with_path['EXAM_COMPLETED_DATE']
).dt.days
biopsy_with_path = (
    biopsy_with_path
    .sort_values('days_diff')
    .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'], keep='first')
    .drop(columns='days_diff')
)

# STEP 1.2: Non-biopsy findings → assign opposite side,
#           but ONLY when exactly one biopsy side exists (L+R = ambiguous → skip)
birads_no_biopsy = birads[~birads['needs_biopsy']].copy()

biopsy_sides = (
    biopsy_with_path[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE']]
    .drop_duplicates()
    .rename(columns={'SIDE': 'biopsy_side'})
)
single_side_accessions = (
    biopsy_sides
    .groupby(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'])
    .filter(lambda g: len(g) == 1)
)

birads_no_biopsy = pd.merge(
    birads_no_biopsy, single_side_accessions,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], how='left'
)
birads_no_biopsy['SIDE'] = birads_no_biopsy['biopsy_side'].map({'L': 'R', 'R': 'L'})
birads_no_biopsy = birads_no_biopsy.drop(columns='biopsy_side')

# STEP 1.3: Rows still without SIDE, accession has NO biopsy at all,
#           and duplicate_count >= 2 → bilateral exam → duplicate as L and R
biopsy_accession_keys = biopsy_sides[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER']].drop_duplicates()

no_side_rows  = birads_no_biopsy[birads_no_biopsy['SIDE'].isna()].copy()
has_side_rows = birads_no_biopsy[birads_no_biopsy['SIDE'].notna()].copy()

no_side_rows = no_side_rows.merge(
    biopsy_accession_keys, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'],
    how='left', indicator=True
)
truly_no_biopsy  = no_side_rows['_merge'] == 'left_only'
no_side_rows     = no_side_rows.drop(columns='_merge')

bilateral_mask = truly_no_biopsy & (no_side_rows['duplicate_count'] >= 2)
bilateral      = no_side_rows[bilateral_mask]
rest           = no_side_rows[~bilateral_mask]

bilateral_L = bilateral.copy(); bilateral_L['SIDE'] = 'L'
bilateral_R = bilateral.copy(); bilateral_R['SIDE'] = 'R'

birads_no_biopsy = pd.concat(
    [has_side_rows, bilateral_L, bilateral_R, rest], ignore_index=True
)

birads_with_path = pd.concat([biopsy_with_path, birads_no_biopsy], ignore_index=True)

In [290]:
# ── STEP 2: Connect to DICOM images (unique_patient_dbt) ─────────────────────

unique_patient_dbt = unique_patient_dbt.rename(columns={'Side': 'SIDE'})

has_side = (birads_with_path[birads_with_path['SIDE'].notna()]
            .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'], keep='first'))
no_side  = (birads_with_path[birads_with_path['SIDE'].isna()]
            .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], keep='first')
            .drop(columns='SIDE'))

# 1. Sided match
merge_sided = pd.merge(
    unique_patient_dbt, has_side,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'], how='inner'
)

# 2. No-side match — only DICOM rows whose accession wasn't already matched above
sided_accessions = merge_sided[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER']].drop_duplicates()
dicom_unmatched  = unique_patient_dbt.merge(
    sided_accessions, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'],
    how='left', indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

merge_no_side = pd.merge(
    dicom_unmatched, no_side,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], how='inner'
)

# 3. DICOM rows with no birads match at all → keep with NaN birads columns
matched_keys = pd.concat([
    merge_sided[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE']],
    merge_no_side[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE']]
]).drop_duplicates()

no_birads = unique_patient_dbt.merge(
    matched_keys, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'],
    how='left', indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

df_final = pd.concat([merge_sided, merge_no_side, no_birads], ignore_index=True)
df_final = df_final.sort_values(['PATIENT_STUDY_ID', 'StudyDate', 'ACCESSION_NUMBER', 'SIDE'], ignore_index=True).drop(columns=['duplicate_count', 'needs_biopsy'])

print(f"unique_patient_dbt : {unique_patient_dbt.shape[0]} rows")
print(f"  ↳ with side      : {len(merge_sided)}")
print(f"  ↳ without side   : {len(merge_no_side)}")
print(f"  ↳ no birads      : {len(no_birads)}")
print(f"df_final           : {df_final.shape[0]} rows")

unique_patient_dbt : 17047 rows
  ↳ with side      : 12244
  ↳ without side   : 4285
  ↳ no birads      : 518
df_final           : 17047 rows


In [291]:
df_final.loc[df_final['FINDING_CATEGORY'].isna(), 'PATIENT_STUDY_ID'].nunique()

251

In [292]:
unique_patient_dbt[unique_patient_dbt['PATIENT_STUDY_ID']==4339601084]

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series
17042,4339601084,46.0,77038224,2018-11-07,DIAG,L,DBT
17043,4339601084,46.0,77038224,2018-11-07,DIAG,R,DBT
17044,4339601084,46.0,65758752,2019-05-16,DIAG,R,DBT
17045,4339601084,47.0,60049293,2020-04-09,SCREEN,L,NaN
17046,4339601084,47.0,60049293,2020-04-09,SCREEN,R,NaN


In [293]:
df_final[df_final['PATIENT_STUDY_ID']==4339601084]

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS
17042,4339601084,46.0,77038224,2018-11-07,DIAG,L,DBT,NaN,NaN,NaN,NaT,NaN,NaT,NaN
17043,4339601084,46.0,77038224,2018-11-07,DIAG,R,DBT,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2018-11-07,427183.0,2018-11-14,Benign
17044,4339601084,46.0,65758752,2019-05-16,DIAG,R,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2019-05-16,NaN,NaT,NaN
17045,4339601084,47.0,60049293,2020-04-09,SCREEN,L,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-04-09,NaN,NaT,NaN
17046,4339601084,47.0,60049293,2020-04-09,SCREEN,R,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-04-09,NaN,NaT,NaN


In [294]:
birads[birads['PATIENT_STUDY_ID']==4339601084]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,needs_biopsy
87049,4339601084,70648957,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2017-10-11,2,False
87050,4339601084,79474989,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,U-Ultrasound,2017-11-07,1,False
87051,4339601084,79474989,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2017-11-07,1,False
87052,4339601084,79253799,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2017-11-16,1,True
87053,4339601084,78102132,Heterogeneously dense (51% - 75%),3 - Probably benign - short interval follow-up,F-Follow-up at short interval (1-11 months),2018-05-24,1,False
87054,4339601084,77038224,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2018-11-07,1,True
87055,4339601084,65758752,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2019-05-16,1,False
87056,4339601084,60049293,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-04-09,2,False
87057,4339601084,67077961,Scattered fibroglandular (25% - 50%),2 - Benign finding,N-Normal interval follow-up,2021-04-15,2,False
87058,4339601084,452833482,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2022-04-25,2,False


In [295]:
pathology[pathology['PATIENT_STUDY_ID']==4339601084]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
11297,4339601084,436123,2017-11-22,Benign,R
11298,4339601084,436122,2017-11-22,Benign,R
11299,4339601084,436121,2017-11-22,Benign,R
11300,4339601084,427183,2018-11-14,Benign,R


In [296]:
df_step1 = df_final.copy()

In [297]:
df_step1.shape

(17047, 14)

In [298]:
df_step1["EXAM_COMPLETED_DATE"] = df_step1["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")
df_step1["PATHOLOGY_DATE"] = df_step1["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

### <span style="color:#FF6347;">**SAVE**</span> file

In [299]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "cancer_cohort" + ".xlsx")
df_step1.to_excel(output_file, index=False)

## **STEP 2**. <span style="color:blue;">**MO cancer**</span>: Label  "Index" = <span style="color:#8A2BE2;">**INDEX**</span> or <span style="color:#00BFFF;">**INDEX-1**</span>
### (OUTPUT) cancer_cohort_index

### <span style="color:#FF6347;">**READ**</span> file (cancer_cohort)

In [300]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, "cancer_cohort" + ".xlsx")
cancer_cohort = pd.read_excel(output_file)

In [301]:
cancer_cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(cancer_cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")
cancer_cohort["PATHOLOGY_DATE"] = pd.to_datetime(cancer_cohort["PATHOLOGY_DATE"], format="%Y-%m-%d")

In [302]:
cancer_cohort["PATIENT_STUDY_ID"].nunique()

2433

In [303]:
cancer_cohort.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS
0,4330066079,30.0,61259016,2020-01-14,DIAG,L,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-01-14,NaN,NaT,NaN
1,4330066079,30.0,61259016,2020-01-14,DIAG,R,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-01-14,NaN,NaT,NaN
2,4330116791,56.0,62335422,2019-09-24,DIAG,L,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN
3,4330116791,56.0,61499674,2019-12-17,DIAG,L,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2019-12-17,NaN,NaT,NaN
4,4330313855,37.0,77236131,2018-05-30,DIAG,L,DBT,Extremely dense (>75%),4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2018-05-30,415556.0,2018-06-05,Benign


## 1. Locate <span style="color:blue;">**Cancer**</span> year, "Index" = <span style="color:#8A2BE2;">**INDEX**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [304]:
# ── Find earliest Malignant pathology per patient → INDEX date ───────────────

cohort = cancer_cohort.copy()
cohort['Index'] = None

# 1. Find the earliest Malignant PATHOLOGY_DATE per patient
first_malignant = (
    cohort[cohort['LESION_CLASS'] == 'Malignant']
    .sort_values('PATHOLOGY_DATE')
    .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first')
    [['PATIENT_STUDY_ID', 'PATHOLOGY_DATE']]
    .rename(columns={'PATHOLOGY_DATE': 'target pathology date'})
)

cohort = cohort.merge(first_malignant, on='PATIENT_STUDY_ID', how='left')

# 2. Mark exams within 6 months BEFORE (or on) the index pathology date
six_months = pd.Timedelta(days=365 / 12 * 6)

cohort['path to exam diff'] = cohort['target pathology date'] - cohort['EXAM_COMPLETED_DATE']

mask_index_cohort = (
    (cohort['path to exam diff'] >= pd.Timedelta(days=0)) &
    (cohort['path to exam diff'] <= six_months)
)

cohort.loc[mask_index_cohort, 'Index'] = 'INDEX'

print(f"✅ Marked {cohort['Index'].eq('INDEX').sum()} rows as 'INDEX'.")

✅ Marked 1128 rows as 'INDEX'.


In [305]:
cohort.reset_index(drop=True, inplace=True)

In [306]:
cohort

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,target pathology date,path to exam diff
0,4330066079,30.0,61259016,2020-01-14,DIAG,L,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-01-14,NaN,NaT,NaN,None,NaT,NaT
1,4330066079,30.0,61259016,2020-01-14,DIAG,R,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-01-14,NaN,NaT,NaN,None,NaT,NaT
2,4330116791,56.0,62335422,2019-09-24,DIAG,L,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,None,NaT,NaT
3,4330116791,56.0,61499674,2019-12-17,DIAG,L,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2019-12-17,NaN,NaT,NaN,None,NaT,NaT
4,4330313855,37.0,77236131,2018-05-30,DIAG,L,DBT,Extremely dense (>75%),4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2018-05-30,415556.0,2018-06-05,Benign,None,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17042,4339601084,46.0,77038224,2018-11-07,DIAG,L,DBT,NaN,NaN,NaN,NaT,NaN,NaT,NaN,None,NaT,NaT
17043,4339601084,46.0,77038224,2018-11-07,DIAG,R,DBT,Heterogeneously dense (51% - 75%),"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2018-11-07,427183.0,2018-11-14,Benign,None,NaT,NaT
17044,4339601084,46.0,65758752,2019-05-16,DIAG,R,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2019-05-16,NaN,NaT,NaN,None,NaT,NaT
17045,4339601084,47.0,60049293,2020-04-09,SCREEN,L,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2020-04-09,NaN,NaT,NaN,None,NaT,NaT


In [307]:
cancer_patients = set(cohort.loc[
    (cohort['LESION_CLASS'] == 'Malignant'),
    'PATIENT_STUDY_ID'
].unique())

cancer = cohort[cohort['PATIENT_STUDY_ID'].isin(cancer_patients)].copy()
print(f"# of cancer patients: {len(cancer_patients)}")
print(f"cancer rows      : {len(cancer)}")
print(f"Index breakdown:\n{cancer['Index'].value_counts(dropna=False)}")

# of cancer patients: 522
cancer rows      : 3805
Index breakdown:
Index
None     2677
INDEX    1128
Name: count, dtype: int64


## 2. Locate <span style="color:blue;">**MO cancer**</span> year, "Index" = <span style="color:#00BFFF;">**INDEX-1**</span>. Adjust <span style="color:pink;"> **screening interval**</span> 

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [308]:
cohort = cancer.copy()

In [309]:
# ── Find INDEX-1: exam between 9–18 months before earliest INDEX exam ─────────
min_months     = pd.Timedelta(days=365 / 12 * 0)
max_months = pd.Timedelta(days=365 / 12 * 30)

In [310]:
# 1. Get the earliest INDEX exam date per patient
earliest_index = (
    cohort[cohort['Index'] == 'INDEX']
    .groupby('PATIENT_STUDY_ID')['EXAM_COMPLETED_DATE']
    .min()
    .reset_index()
    .rename(columns={'EXAM_COMPLETED_DATE': 'earliest index date'})
)

cohort = cohort.merge(earliest_index, on='PATIENT_STUDY_ID', how='left')

# 2. Time between each exam and the earliest INDEX exam
cohort['exam to index diff'] = cohort['earliest index date'] - cohort['EXAM_COMPLETED_DATE']

# 3. Window: at least 9 months before, no more than 18 months before
mask_window = (
    (cohort['exam to index diff'] > min_months) &
    (cohort['exam to index diff'] <= max_months)
)

cohort.loc[mask_window, 'Index'] = 'INDEX-1'

# # 4. Among candidates, keep the one closest to INDEX (largest exam date = smallest diff)
# index_minus_1_idx = (
#     cohort[mask_window]
#     .sort_values('exam to index diff')                          # smallest diff first
#     .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first') # closest per patient
#     .index
# )

# cohort.loc[index_minus_1_idx, 'Index'] = 'INDEX-1'

print(f"✅ Marked {cohort['Index'].eq('INDEX-1').sum()} rows as 'INDEX-1'.")
print(f"   Patients with INDEX-1 : {cohort[cohort['Index'] == 'INDEX-1']['PATIENT_STUDY_ID'].nunique()}")
print(f"   Patients with INDEX   : {cohort[cohort['Index'] == 'INDEX']['PATIENT_STUDY_ID'].nunique()}")

✅ Marked 581 rows as 'INDEX-1'.
   Patients with INDEX-1 : 221
   Patients with INDEX   : 522


In [311]:
cohort[cohort['Index'] == 'INDEX-1']['PATIENT_STUDY_ID'].unique()

array([4333002438, 4333003087, 4333004303, 4333004761, 4333010687,
       4333010697, 4333011821, 4333012288, 4333012872, 4333012936,
       4333013763, 4333017194, 4333020276, 4333023156, 4333023918,
       4333024028, 4333025059, 4333028820, 4333041185, 4333062996,
       4333063695, 4333065693, 4333065807, 4333065960, 4333073161,
       4333076307, 4333078057, 4333081422, 4333081537, 4333086135,
       4333089665, 4333090055, 4333090748, 4333093019, 4333094101,
       4333094910, 4333095097, 4333096339, 4333096584, 4333097279,
       4333097425, 4333100489, 4333104624, 4333105780, 4333105901,
       4333111514, 4333114943, 4333120230, 4333127268, 4333129055,
       4333133147, 4333135321, 4333136476, 4333144900, 4333150327,
       4333150433, 4333154217, 4333157062, 4333159108, 4333161902,
       4333165191, 4333166790, 4333168819, 4333178521, 4333180830,
       4333180863, 4333182949, 4333183457, 4333184368, 4333189732,
       4333198219, 4333203048, 4333205138, 4333205689, 4333213

In [312]:
cohort[cohort['PATIENT_STUDY_ID'] == 4333010687]

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,target pathology date,path to exam diff,earliest index date,exam to index diff
86,4333010687,64.0,77464018,2018-07-07,SCREEN,L,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2018-07-07,NaN,NaT,NaN,INDEX-1,2020-02-17,590 days,2019-12-21,532 days
87,4333010687,64.0,77464018,2018-07-07,SCREEN,R,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2018-07-07,NaN,NaT,NaN,INDEX-1,2020-02-17,590 days,2019-12-21,532 days
88,4333010687,65.0,62010914,2019-12-21,SCREEN,L,DBT,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2019-12-21,NaN,NaT,NaN,INDEX,2020-02-17,58 days,2019-12-21,0 days
89,4333010687,65.0,62010914,2019-12-21,SCREEN,R,DBT,Heterogeneously dense (51% - 75%),0 - Need additional imaging evaluation,P-Additional projections,2019-12-21,NaN,NaT,NaN,INDEX,2020-02-17,58 days,2019-12-21,0 days
90,4333010687,65.0,61024786,2020-01-17,DIAG,L,NaN,Heterogeneously dense (51% - 75%),4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2020-01-17,481678.0,2020-02-17,Malignant,INDEX,2020-02-17,31 days,2019-12-21,-27 days
91,4333010687,65.0,61024786,2020-01-17,NaN,L,DBT,Heterogeneously dense (51% - 75%),4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2020-01-17,481678.0,2020-02-17,Malignant,INDEX,2020-02-17,31 days,2019-12-21,-27 days
92,4333010687,66.0,67980187,2021-02-17,DIAG,R,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2021-02-17,NaN,NaT,NaN,None,2020-02-17,-366 days,2019-12-21,-424 days
93,4333010687,67.0,453985357,2022-04-06,SCREEN,R,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2022-04-06,NaN,NaT,NaN,None,2020-02-17,-779 days,2019-12-21,-837 days


In [313]:
cohort = cohort.drop(columns = ["target pathology date", "path to exam diff", "earliest index date", "exam to index diff"])

In [314]:
cohort.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index
0,4330325599,38.0,75132824,2016-04-20,DIAG,L,DBT,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2016-04-20,455315.0,2016-04-20,Malignant,INDEX
1,4330325599,38.0,75132824,2016-04-20,DIAG,R,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2016-04-20,NaN,NaT,NaN,INDEX
2,4330325599,39.0,73785729,2017-04-17,DIAG,R,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2017-04-17,NaN,NaT,NaN,None
3,4330430711,81.0,62364668,2019-09-18,DIAG,L,DBT,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2019-09-18,482650.0,2019-10-10,Malignant,INDEX
4,4333000414,56.0,71446941,2017-05-11,DIAG,L,DBT,Heterogeneously dense (51% - 75%),4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2017-05-11,444858.0,2017-05-19,Malignant,INDEX


### <span style="color:#FF6347;">**SAVE**</span> file (cancer_cohort)

In [315]:
cohort["EXAM_COMPLETED_DATE"] = cohort["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")
cohort["PATHOLOGY_DATE"] = cohort["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

In [316]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'cancer_cohort' + ".xlsx")
cohort.to_excel(output_file, index=False)

## **STEP 3**. Extract <span style="color:blue;">**MO cancer**</span> cohort only
### (Output) mo_cancer, mo_cancer_cohort

#### <span style="color:#FF6347;">**READ**</span> file (cancer_cohort)

In [317]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort_index' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'cancer_cohort' + ".xlsx")
cohort = pd.read_excel(output_file)

In [318]:
cohort

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index
0,4330325599,38.0,75132824,2016-04-20,DIAG,L,DBT,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2016-04-20,455315.0,2016-04-20,Malignant,INDEX
1,4330325599,38.0,75132824,2016-04-20,DIAG,R,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2016-04-20,NaN,NaN,NaN,INDEX
2,4330325599,39.0,73785729,2017-04-17,DIAG,R,DBT,Heterogeneously dense (51% - 75%),1 - Negative,N-Normal interval follow-up,2017-04-17,NaN,NaN,NaN,NaN
3,4330430711,81.0,62364668,2019-09-18,DIAG,L,DBT,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2019-09-18,482650.0,2019-10-10,Malignant,INDEX
4,4333000414,56.0,71446941,2017-05-11,DIAG,L,DBT,Heterogeneously dense (51% - 75%),4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2017-05-11,444858.0,2017-05-19,Malignant,INDEX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3800,4335987312,75.0,71114169,2018-04-26,DIAG,R,NaN,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2018-04-26,NaN,NaN,NaN,NaN
3801,4336131298,77.0,77177026,2018-07-05,NaN,L,NaN,Heterogeneously dense (51% - 75%),"5 - Highly suggestive of malignancy, appropria...",B-Biopsy should be considered,2018-07-05,411322.0,2018-07-11,Malignant,INDEX
3802,4336131298,78.0,64650920,2019-07-12,DIAG,L,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-07-12,NaN,NaN,NaN,NaN
3803,4336131298,78.0,64650920,2019-07-12,DIAG,R,DBT,Heterogeneously dense (51% - 75%),2 - Benign finding,N-Normal interval follow-up,2019-07-12,NaN,NaN,NaN,NaN


In [319]:
cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")
cohort["PATHOLOGY_DATE"] = pd.to_datetime(cohort["PATHOLOGY_DATE"], format="%Y-%m-%d")

### **Criteria**:
#### 1. At <span style="color:#8A2BE2;">**INDEX**</span>, "LESION_CLASS" = "Malignant"
##### no need to check as all patients in cancer_cohort already meet this criteria
#### 2. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Series" = "DBT"
#### 3. At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 4. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Study" = "SCREEN"

In [320]:
normal_cats = ['1 - Negative', '2 - Benign finding']

recall_cats = [
    '0 - Need additional imaging evaluation',
    '3 - Probably benign - short interval follow-up',
    '4 - Suspicious abnormality, biopsy should be considered',
    '4A - Suspicious abnormality - biopsy should be considered - low suspicion',
    '4B - Suspicious abnormality - biopsy should be considered - intermediate suspicion',
    '4C - Suspicious abnormality - biopsy should be considered - moderate suspicion',
    '5 - Highly suggestive of malignancy, appropriate action should be taken',
    '6 - Known biopsy proven malignancy'
]

In [321]:
# Patients with ANY recall finding at INDEX-1 (disqualify, takes priority)
recall_patients = set(cohort.loc[
    (cohort['Index'] == 'INDEX-1') &
    (cohort['Series'] == 'DBT') &
    (cohort['FINDING_CATEGORY'].isin(recall_cats)) &
    (cohort['Study'] == 'SCREEN'),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX-1
normal_patients = set(cohort.loc[
    (cohort['Index'] == 'INDEX-1') &
    (cohort['Series'] == 'DBT') &
    (cohort['FINDING_CATEGORY'].isin(normal_cats)) &
    (cohort['Study'] == 'SCREEN'),
    'PATIENT_STUDY_ID'
].unique())

# Exclude wins: remove any patient who appeared in recall
qualifying_patients = normal_patients - recall_patients

print(f"Normal patients     : {len(normal_patients)}")
print(f"Recall patients     : {len(recall_patients)}")
print(f"Overlap removed     : {len(normal_patients & recall_patients)}")
print(f"Qualifying patients : {len(qualifying_patients)}")

mo_cancer = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].copy()
print(f"mo_cancer rows      : {len(mo_cancer)}")
print(f"Index breakdown:\n{mo_cancer['Index'].value_counts(dropna=False)}")

Normal patients     : 114
Recall patients     : 17
Overlap removed     : 3
Qualifying patients : 111
mo_cancer rows      : 961
Index breakdown:
Index
NaN        383
INDEX-1    318
INDEX      260
Name: count, dtype: int64


In [322]:
cohort_candidate = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].reset_index(drop=True)
cohort_candidate.sort_values(['PATIENT_STUDY_ID', 'StudyDate', 'ACCESSION_NUMBER']).reset_index(drop=True, inplace=True)

cohort_candidate["EXAM_COMPLETED_DATE"] = cohort_candidate["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")
cohort_candidate["PATHOLOGY_DATE"] = cohort_candidate["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

##### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_cohort**

In [323]:
mo_cancer_cohort = cohort_candidate.copy()

In [324]:
mo_cancer_cohort['PATIENT_STUDY_ID'].nunique()

111

In [325]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'mo_cancer_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'mo_cancer_cohort' + ".xlsx")

mo_cancer_cohort.to_excel(output_file, index=False)

##### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer** (only retain INDX & INDEX-1)

In [326]:
mo_cancer= mo_cancer_cohort[(mo_cancer_cohort["Index"]=="INDEX") | (mo_cancer_cohort["Index"]=="INDEX-1")]

In [327]:
mo_cancer['PATIENT_STUDY_ID'].nunique()

111

In [328]:
# output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'mo_cancer' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'mo_cancer' + ".xlsx")

mo_cancer.to_excel(output_file, index=False)

### **Broader Criteria**:
#### 1. At <span style="color:#8A2BE2;">**INDEX**</span>, "LESION_CLASS" = "Malignant"
##### no need to check as all patients in cancer_cohort already meet this criteria
#### 2. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Series" = "DBT"
#### 3. At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"

In [329]:
# Patients with ANY recall finding at INDEX-1 (disqualify, takes priority)
recall_patients = set(cohort.loc[
    (cohort['Index'] == 'INDEX-1') &
    (cohort['Series'] == 'DBT') &
    (cohort['FINDING_CATEGORY'].isin(recall_cats)),
    'PATIENT_STUDY_ID'
].unique())

# Patients with a normal finding at INDEX-1
normal_patients = set(cohort.loc[
    (cohort['Index'] == 'INDEX-1') &
    (cohort['Series'] == 'DBT') &
    (cohort['FINDING_CATEGORY'].isin(normal_cats)),
    'PATIENT_STUDY_ID'
].unique())

# Exclude wins: remove any patient who appeared in recall
qualifying_patients = normal_patients - recall_patients

print(f"Normal patients     : {len(normal_patients)}")
print(f"Recall patients     : {len(recall_patients)}")
print(f"Overlap removed     : {len(normal_patients & recall_patients)}")
print(f"Qualifying patients : {len(qualifying_patients)}")

mo_cancer = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].copy()
print(f"mo_cancer rows      : {len(mo_cancer)}")
print(f"Index breakdown:\n{mo_cancer['Index'].value_counts(dropna=False)}")

Normal patients     : 168
Recall patients     : 23
Overlap removed     : 10
Qualifying patients : 158
mo_cancer rows      : 1322
Index breakdown:
Index
NaN        524
INDEX-1    431
INDEX      367
Name: count, dtype: int64


In [330]:
cohort_candidate = cohort[cohort['PATIENT_STUDY_ID'].isin(qualifying_patients)].reset_index(drop=True)
cohort_candidate.sort_values(['PATIENT_STUDY_ID', 'StudyDate', 'ACCESSION_NUMBER']).reset_index(drop=True, inplace=True)

cohort_candidate["EXAM_COMPLETED_DATE"] = cohort_candidate["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")
cohort_candidate["PATHOLOGY_DATE"] = cohort_candidate["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

##### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_candidate_cohort**

In [331]:
mo_cancer_cohort = cohort_candidate.copy()

In [332]:
mo_cancer_cohort['PATIENT_STUDY_ID'].nunique()

158

In [333]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'mo_cancer_candidate_cohort' + ".xlsx")
output_file = os.path.join("/Users/tracyliu/Library/CloudStorage/OneDrive-UniversityofPittsburgh/R01-MO-DBT/MO-DBT-data-curation/Data", study, 'mo_cancer_candidate_cohort' + ".xlsx")

mo_cancer_cohort.to_excel(output_file, index=False)